In [ ]:
import numpy as np
from osgeo import gdal
import gma

dsin = gdal.Open('/Users/jeong/MIMC3/examples/example_chunli_202601/20020805152936_AST1_B2_sub.tif', gdal.GA_ReadOnly)
geotransform = dsin.GetGeoTransform()
raster_shape = (dsin.RasterYSize, dsin.RasterXSize)
proj = dsin.GetProjection()
dsin = None  # Close the dataset

In [ ]:
print(geotransform)
print(raster_shape)

In [ ]:
u_min = 2660
v_min = 2620
u_max = 3330
v_max = 3300
du = 10
dv = -10


dx = -3280477.340 + 3280380.035
dy = 350633.267 - 350456.348

vx = dx / 28.0 * 365.25
vy = dy / 28.0 * 365.25

vec_u = list(range(u_min, u_max+1, du))
vec_v = list(range(v_max, v_min-1, dv))
num_pts = len(vec_u) * len(vec_v)
print(f"Number of points: {num_pts}")

xyuvav_arr = np.zeros((num_pts, 6), dtype='float64')

for idx_v, v in enumerate(vec_v):
    for idx_u, u in enumerate(vec_u):

        x = geotransform[0] + u * geotransform[1] + v * geotransform[2]
        y = geotransform[3] + u * geotransform[4] + v * geotransform[5]
        print(f"{x}, {y}, {u}, {v}")

        xyuvav_arr[idx_v * len(vec_u) + idx_u, :] = [x, y, u, v, vx, vy]

In [ ]:
print(xyuvav_arr[:10, :])
gma.write(xyuvav_arr, '/Users/jeong/MIMC3/examples/example_chunli_202601/xyuvav_arr.gma', dtype='float64')

In [ ]:
# MIMC3 [first tiff image] [second tiff image] [xyuvav file] [output directory]

'''
./MIMC3 \
    /Users/jeong/MIMC3/examples/example_chunli_202601/20020805152936_AST1_B2_sub.tif \
    /Users/jeong/MIMC3/examples/example_chunli_202601/20120908121616_AST1_B2_sub.tif \
    /Users/jeong/MIMC3/examples/example_chunli_202601/xyuvav_arr.gma \
    /Users/jeong/MIMC3/examples/example_chunli_202601/output
'''
import matplotlib.pyplot as plt

vx = gma.read('/Users/jeong/MIMC3/examples/example_chunli_202601/output/vmap_20020805152936_20120908121616_vx.GMA')
vy = gma.read('/Users/jeong/MIMC3/examples/example_chunli_202601/output/vmap_20020805152936_20120908121616_vy.GMA')
qual = gma.read('/Users/jeong/MIMC3/examples/example_chunli_202601/output/vmap_20020805152936_20120908121616_qual.GMA')

In [ ]:
fig = plt.figure(figsize=(15,5))
ax = fig.add_subplot(1,3,1)
ax.imshow(vx, cmap='jet')
ax = fig.add_subplot(1,3,2)
ax.imshow(vy, cmap='jet')
ax = fig.add_subplot(1,3,3)
ax.imshow(qual, cmap='jet')


In [ ]:
vx = gma.read('/Users/jeong/MIMC3/examples/example_chunli_202601/output/vmap_20020805152936_20120908121616_vx.GMA')
vy = gma.read('/Users/jeong/MIMC3/examples/example_chunli_202601/output/vmap_20020805152936_20120908121616_vy.GMA')
qual = gma.read('/Users/jeong/MIMC3/examples/example_chunli_202601/output/vmap_20020805152936_20120908121616_qual.GMA')

arrout = np.array([vx, vy, qual])
x0 = geotransform[0] + u_min * geotransform[1] + v_max * geotransform[2]
y0 = geotransform[3] + u_min * geotransform[4] + v_max * geotransform[5]
pixel_width = geotransform[1] * du
pixel_height = geotransform[5] * (dv)
new_geotransform = (x0, pixel_width, 0, y0, 0, pixel_height)

drvout = gdal.GetDriverByName('GTiff')
dsout = drvout.Create('/Users/jeong/MIMC3/examples/example_chunli_202601/output/vmap_20020805152936_20120908121616_output.tif',
                      vx.shape[1], vx.shape[0], 3, gdal.GDT_Float32)
dsout.SetGeoTransform(new_geotransform)
dsout.SetProjection(proj)
dsout.GetRasterBand(1).WriteArray(vx)
dsout.GetRasterBand(2).WriteArray(vy)
dsout.GetRasterBand(3).WriteArray(qual)
dsout.FlushCache()
dsout = None


vxf = vx.copy()
vxf[qual<0.4] = np.nan
vyf = vy.copy()
vyf[qual<0.4] = np.nan
dsout = drvout.Create('/Users/jeong/MIMC3/examples/example_chunli_202601/output/vmap_20020805152936_20120908121616_output_filtered.tif',
                      vx.shape[1], vx.shape[0], 3, gdal.GDT_Float32)
dsout.SetGeoTransform(new_geotransform)
dsout.SetProjection(proj)
dsout.GetRasterBand(1).WriteArray(vxf)
dsout.GetRasterBand(2).WriteArray(vyf)
dsout.FlushCache()
dsout = None